# SNU AI Challenge — Full 24-Class Colab Pipeline

This notebook starts from the original 9,535 labeled training samples. It includes all 1,478 `No_ordering=True` samples whose target is `[1, 2, 3, 4]`. No result or cache from the previous 8,057-row ordering-only run is reused.

Run the cells from top to bottom using a VS Code Colab GPU kernel or Google Colab.

## 1. Goal and assumptions

The notebook extracts the raw-data bundle, validates the MD5 group-stratified 24-class split, computes frozen ConvNeXt/Qwen representations, runs MD5 group-stratified five-fold validation, and finally trains one scorer on all 9,535 rows. Feature-cache shards and checkpoints are persisted under `MyDrive/snu ai/`.

Use only the provided competition training data. Do not place test data in this training bundle.

## 2. Parameters

For parallel Colab runtimes, assign disjoint fold lists. Never run the same fold in two runtimes writing to the same result directory.

In [1]:
import os
from pathlib import Path

DRIVE_ROOT = Path(os.environ.get("SNU_DRIVE_ROOT", "/content/drive/MyDrive/snu ai"))
BUNDLE_PATH = DRIVE_ROOT / "snu_colab_full_train_bundle.tar"
CODE_BUNDLE_PATH = DRIVE_ROOT / "snu_colab_resident_training_code.tar"
WORKDIR = Path(os.environ.get("SNU_WORKDIR", "/content/snu_ai"))
GROUP_FOLDS_CSV = WORKDIR / "snuaichallenge_data/splits/group_stratified_folds_5.csv"
DRIVE_GROUP_FOLDS_CSV = DRIVE_ROOT / "splits/group_stratified_folds_5.csv"
DRIVE_CACHE_SHARDS = DRIVE_ROOT / "cache/full_train_9535_shards"
DRIVE_RESULTS_ROOT = DRIVE_ROOT / "results/full_24class_group_stratified_v1"
LOCAL_CACHE_ROOT = WORKDIR / "feature_cache/full_train_9535"

FOLDS_TO_RUN = [0, 1, 2, 3, 4]
CACHE_SHARD_SIZE = 250
TRAIN_FINAL_SCORER = True

HIDDEN_DIM = 256
VERIFIER = "transformer"
BATCH_SIZE = 128
EVALUATION_BATCH_SIZE = 256
MAX_EPOCHS = 20
PATIENCE = 4
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
TOP_H = 5
TOP_M = 8
ALPHA, BETA, GAMMA, DELTA = 0.1, 1.0, 1.0, 1.0
SEED = 42

assert FOLDS_TO_RUN and all(fold in range(5) for fold in FOLDS_TO_RUN)
assert len(FOLDS_TO_RUN) == len(set(FOLDS_TO_RUN))
assert CACHE_SHARD_SIZE > 0


## 3. Mount Drive and install dependencies

In [2]:
try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    drive.mount("/content/drive")
else:
    print("google.colab is unavailable; configured paths will be used directly.")

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_CACHE_SHARDS.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS_ROOT.mkdir(parents=True, exist_ok=True)


Mounted at /content/drive


In [3]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "transformers==5.13.0",
        "accelerate==1.14.0",
        "pillow>=10",
    ],
    check=True,
)


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '--quiet', 'transformers==5.13.0', 'accelerate==1.14.0', 'pillow>=10'], returncode=0)

## 4. Extract the raw training bundle locally

The bundle must contain `snu_ordering/`, `train.csv`, all 9,535 training image folders. The MD5 group-stratified split is generated from these raw images and persisted to Drive for reuse.

In [4]:
if not BUNDLE_PATH.is_file():
    raise FileNotFoundError(f"Training bundle not found: {BUNDLE_PATH}")
if not CODE_BUNDLE_PATH.is_file():
    raise FileNotFoundError(f"Optimized code bundle not found: {CODE_BUNDLE_PATH}")

WORKDIR.mkdir(parents=True, exist_ok=True)
bundle_marker = WORKDIR / "snuaichallenge_data/train.csv"
image_marker = WORKDIR / "snuaichallenge_data/train"
if not bundle_marker.is_file() or not image_marker.is_dir():
    subprocess.run(
        ["tar", "-xf", str(BUNDLE_PATH), "-C", str(WORKDIR)],
        check=True,
    )

subprocess.run(
    ["tar", "-xf", str(CODE_BUNDLE_PATH), "-C", str(WORKDIR)],
    check=True,
)
for module_name in list(sys.modules):
    if module_name == "snu_ordering" or module_name.startswith("snu_ordering."):
        del sys.modules[module_name]
import importlib
importlib.invalidate_caches()

print(f"Project root: {WORKDIR}")
print(f"Persistent cache shards: {DRIVE_CACHE_SHARDS}")
print(f"Persistent results: {DRIVE_RESULTS_ROOT}")


Project root: /content/snu_ai
Persistent cache shards: /content/drive/MyDrive/snu ai/cache/full_train_9535_shards
Persistent results: /content/drive/MyDrive/snu ai/results/full_24class_group_stratified_v1


## 5. Build and validate the MD5 group-stratified MD5 group-stratified 24-class split

In [5]:
import json
import random
from collections import Counter

import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before continuing.")

sys.path.insert(0, str(WORKDIR))
from snu_ordering.data import load_samples

import ast
import hashlib
import json
import shutil
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import networkx as nx
import pandas as pd
import sklearn
from sklearn.model_selection import StratifiedGroupKFold

TRAIN_CSV = WORKDIR / "snuaichallenge_data/train.csv"
IMAGE_ROOT = WORKDIR / "snuaichallenge_data/train"
GROUP_SPLIT_SEED = 42
GROUP_SPLIT_FOLDS = 5
MAX_SAMPLES_PER_HASH = 3
GROUP_SPLIT_METHOD = "md5_stratified_group_kfold_v1"


def _compute_md5(path):
    with path.open("rb") as file:
        return hashlib.md5(file.read()).hexdigest()


def _build_md5_group_stratified_folds(train_csv, image_root):
    train = pd.read_csv(train_csv)
    train["answer_tuple"] = train["Answer"].map(
        lambda value: str(ast.literal_eval(value))
    )

    image_records = []
    for _, row in train.iterrows():
        for image_index in range(1, 5):
            image_records.append(
                {
                    "sample_id": row["Id"],
                    "path": image_root / str(row["Id"]) / str(row[f"Input_{image_index}"]),
                }
            )

    print(f"Computing MD5 for {len(image_records)} training images...")
    with ThreadPoolExecutor(max_workers=32) as executor:
        hashes = list(executor.map(_compute_md5, [record["path"] for record in image_records]))
    for record, image_hash in zip(image_records, hashes, strict=True):
        record["hash"] = image_hash

    hash_counter = defaultdict(set)
    for record in image_records:
        hash_counter[record["hash"]].add(record["sample_id"])
    hash_to_samples = {
        image_hash: sample_ids
        for image_hash, sample_ids in hash_counter.items()
        if len(sample_ids) <= MAX_SAMPLES_PER_HASH
    }

    graph = nx.Graph()
    graph.add_nodes_from(train["Id"].unique())
    for sample_ids in hash_to_samples.values():
        sample_ids = list(sample_ids)
        for left in range(len(sample_ids)):
            for right in range(left + 1, len(sample_ids)):
                graph.add_edge(sample_ids[left], sample_ids[right])

    components = list(nx.connected_components(graph))
    id_to_cluster = {
        sample_id: cluster_id
        for cluster_id, component in enumerate(components)
        for sample_id in component
    }
    train["cluster_id"] = train["Id"].map(id_to_cluster)

    splitter = StratifiedGroupKFold(
        n_splits=GROUP_SPLIT_FOLDS,
        shuffle=True,
        random_state=GROUP_SPLIT_SEED,
    )
    train["Fold"] = -1
    for fold, (_, validation_indices) in enumerate(
        splitter.split(
            train,
            train["answer_tuple"],
            groups=train["cluster_id"],
        )
    ):
        train.loc[validation_indices, "Fold"] = fold

    train["split_method"] = GROUP_SPLIT_METHOD
    train["split_seed"] = GROUP_SPLIT_SEED
    train["split_max_samples_per_hash"] = MAX_SAMPLES_PER_HASH
    train["split_sklearn_version"] = sklearn.__version__
    build_summary = {
        "clusters": len(components),
        "excluded_frequent_hashes": sum(
            len(sample_ids) > MAX_SAMPLES_PER_HASH
            for sample_ids in hash_counter.values()
        ),
    }
    return train, build_summary


def _validate_md5_group_stratified_folds(folded, raw_train):
    required = {
        "Id", "Answer", "No_ordering", "answer_tuple", "cluster_id", "Fold",
        "split_method", "split_seed", "split_max_samples_per_hash",
        "split_sklearn_version",
    }
    missing = required.difference(folded.columns)
    if missing:
        raise ValueError(f"Group split is missing columns: {sorted(missing)}")
    if folded["Id"].astype(str).tolist() != raw_train["Id"].astype(str).tolist():
        raise ValueError("Group split IDs/order do not match train.csv")
    if folded["Id"].duplicated().any():
        raise ValueError("Group split contains duplicate IDs")

    expected_answers = raw_train["Answer"].map(lambda value: str(ast.literal_eval(value)))
    if folded["answer_tuple"].astype(str).tolist() != expected_answers.tolist():
        raise ValueError("Group split Answer labels do not match train.csv")
    if set(folded["split_method"].astype(str)) != {GROUP_SPLIT_METHOD}:
        raise ValueError("Unexpected group split method")
    if set(folded["split_seed"].astype(int)) != {GROUP_SPLIT_SEED}:
        raise ValueError("Unexpected group split seed")
    if set(folded["split_max_samples_per_hash"].astype(int)) != {MAX_SAMPLES_PER_HASH}:
        raise ValueError("Unexpected MD5 reuse threshold")

    folded["Fold"] = folded["Fold"].astype(int)
    if set(folded["Fold"]) != set(range(GROUP_SPLIT_FOLDS)):
        raise ValueError("Expected folds 0 through 4")
    cross_fold_clusters = int(
        (folded.groupby("cluster_id")["Fold"].nunique() > 1).sum()
    )
    if cross_fold_clusters:
        raise ValueError(f"{cross_fold_clusters} MD5 clusters cross folds")

    identity_mask = folded["answer_tuple"].eq("[1, 2, 3, 4]")
    no_ordering_mask = folded["No_ordering"].astype(str).str.lower().eq("true")
    if not identity_mask.equals(no_ordering_mask):
        raise ValueError("Identity labels and No_ordering=True rows differ")
    classes_per_fold = folded.groupby("Fold")["answer_tuple"].nunique().to_dict()
    if set(classes_per_fold.values()) != {24}:
        raise ValueError(f"Every fold must contain all 24 classes: {classes_per_fold}")

    return {
        "method": GROUP_SPLIT_METHOD,
        "rows": len(folded),
        "classes": folded["answer_tuple"].nunique(),
        "identity_rows": int(identity_mask.sum()),
        "clusters": folded["cluster_id"].nunique(),
        "cross_fold_clusters": cross_fold_clusters,
        "fold_counts": {
            str(fold): int(count)
            for fold, count in folded["Fold"].value_counts().sort_index().items()
        },
        "built_with_sklearn": str(folded["split_sklearn_version"].iloc[0]),
    }


GROUP_FOLDS_CSV.parent.mkdir(parents=True, exist_ok=True)
DRIVE_GROUP_FOLDS_CSV.parent.mkdir(parents=True, exist_ok=True)
if DRIVE_GROUP_FOLDS_CSV.is_file():
    shutil.copy2(DRIVE_GROUP_FOLDS_CSV, GROUP_FOLDS_CSV)
elif not GROUP_FOLDS_CSV.is_file():
    folded_frame, group_split_build = _build_md5_group_stratified_folds(
        TRAIN_CSV,
        IMAGE_ROOT,
    )
    temporary_csv = GROUP_FOLDS_CSV.with_suffix(".csv.tmp")
    folded_frame.to_csv(temporary_csv, index=False)
    temporary_csv.replace(GROUP_FOLDS_CSV)
    print("group split build:", json.dumps(group_split_build, indent=2))

raw_train_frame = pd.read_csv(TRAIN_CSV)
group_fold_frame = pd.read_csv(GROUP_FOLDS_CSV)
group_split_summary = _validate_md5_group_stratified_folds(
    group_fold_frame,
    raw_train_frame,
)
if not DRIVE_GROUP_FOLDS_CSV.is_file():
    temporary_drive_csv = DRIVE_GROUP_FOLDS_CSV.with_suffix(".csv.tmp")
    shutil.copy2(GROUP_FOLDS_CSV, temporary_drive_csv)
    temporary_drive_csv.replace(DRIVE_GROUP_FOLDS_CSV)
print("group split:", json.dumps(group_split_summary, indent=2))

samples = load_samples(
    GROUP_FOLDS_CSV,
    IMAGE_ROOT,
    require_labels=True,
    validate_images=True,
)

answer_counts = Counter(sample.rank_label for sample in samples)
fold_counts = Counter(sample.fold for sample in samples)
identity = (1, 2, 3, 4)
no_ordering_count = sum(sample.no_ordering is True for sample in samples)
identity_count = answer_counts[identity]

assert len(samples) == 9535
assert len(answer_counts) == 24
assert identity_count == 1478
assert no_ordering_count == 1478
assert set(fold_counts) == set(range(5))
assert sum(fold_counts.values()) == 9535
assert all(sample.fold is not None for sample in samples)

print("torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA capability:", torch.cuda.get_device_capability(0))
print("rows / classes / identity:", len(samples), len(answer_counts), identity_count)
print("fold counts:", dict(sorted(fold_counts.items())))


Computing MD5 for 38140 training images...
group split build: {
  "clusters": 8147,
  "excluded_frequent_hashes": 2081
}
group split: {
  "method": "md5_stratified_group_kfold_v1",
  "rows": 9535,
  "classes": 24,
  "identity_rows": 1478,
  "clusters": 8147,
  "cross_fold_clusters": 0,
  "fold_counts": {
    "0": 1915,
    "1": 1905,
    "2": 1886,
    "3": 1911,
    "4": 1918
  },
  "built_with_sklearn": "1.6.1"
}
torch: 2.11.0+cu128
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
CUDA capability: (12, 0)
rows / classes / identity: 9535 24 1478
fold counts: {0: 1915, 1: 1905, 2: 1886, 3: 1911, 4: 1918}


## 6. Restore persisted feature-cache shards

Cache shards contain 250 samples each. On a new Colab runtime, completed shards are extracted to local storage and only missing samples are encoded.

In [6]:
import tarfile

LOCAL_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
persisted_relative_paths = set()
cache_shards = sorted(DRIVE_CACHE_SHARDS.glob("cache_*.tar"))
for shard_path in cache_shards:
    with tarfile.open(shard_path, "r") as archive:
        members = [member for member in archive.getmembers() if member.isfile()]
        persisted_relative_paths.update(member.name for member in members)
        archive.extractall(LOCAL_CACHE_ROOT)

print(f"restored shards: {len(cache_shards)}")
print(f"restored cache files: {len(persisted_relative_paths)}")


/tmp/ipykernel_6395/1588790026.py:10: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  archive.extractall(LOCAL_CACHE_ROOT)


restored shards: 48
restored cache files: 9535


## 7. Cache all 9,535 frozen representations

Qwen is pinned to revision `37eefda6797047d24d5d08184aea3e097f3f8a50`. Ampere-or-newer GPUs use BF16; T4 uses FP16. The cache step is resumable from Drive shards.

In [7]:
import gc
import shutil
import time

from snu_ordering.data import load_sample_images
from snu_ordering.features import ConvNeXtTinyFeatureExtractor
from snu_ordering.qwen_representations import (
    DEFAULT_MAX_PIXELS,
    DEFAULT_MIN_PIXELS,
    FrozenQwenRepresentationExtractor,
    QWEN_MODEL_ID,
    QWEN_MODEL_REVISION,
    build_structured_representation,
    cache_path_for,
    load_representation,
    representation_fingerprint,
    save_representation,
)
from snu_ordering.text import chunk_caption

existing_indices = [int(path.stem.split("_")[-1]) for path in cache_shards]
next_shard_index = max(existing_indices, default=-1) + 1

def persist_cache_paths(paths):
    global next_shard_index
    if not paths:
        return
    local_tar = WORKDIR / f"cache_{next_shard_index:04d}.tar"
    final_tar = DRIVE_CACHE_SHARDS / local_tar.name
    temporary_tar = DRIVE_CACHE_SHARDS / f"{local_tar.name}.tmp"
    with tarfile.open(local_tar, "w") as archive:
        for path in paths:
            archive.add(path, arcname=path.relative_to(LOCAL_CACHE_ROOT).as_posix())
    shutil.copy2(local_tar, temporary_tar)
    temporary_tar.replace(final_tar)
    local_tar.unlink()
    print(f"persisted {len(paths)} files -> {final_tar.name}", flush=True)
    next_shard_index += 1

pending = []
unpersisted_existing = []
for sample in samples:
    chunks = chunk_caption(sample.caption)
    fingerprint = representation_fingerprint(
        sample,
        chunks,
        model_id=QWEN_MODEL_ID,
        revision=QWEN_MODEL_REVISION,
        min_pixels=DEFAULT_MIN_PIXELS,
        max_pixels=DEFAULT_MAX_PIXELS,
    )
    cache_path = cache_path_for(LOCAL_CACHE_ROOT, sample.sample_id)
    valid = False
    if cache_path.is_file():
        try:
            cached = load_representation(cache_path)
            hidden_size = cached.full_caption_feature.shape[0]
            valid = (
                cached.sample_id == sample.sample_id
                and cached.full_caption == chunks["full_caption"]
                and cached.event_units == tuple(chunks["event_units"])
                and cached.convnext_frame_features.ndim == 2
                and cached.convnext_frame_features.shape[0] == 4
                and cached.qwen_frame_features.shape == (4, hidden_size)
                and cached.event_unit_features.shape
                == (len(chunks["event_units"]), hidden_size)
            )
        except (ValueError, RuntimeError, EOFError):
            valid = False
    if valid:
        relative_name = cache_path.relative_to(LOCAL_CACHE_ROOT).as_posix()
        if relative_name not in persisted_relative_paths:
            unpersisted_existing.append(cache_path)
    else:
        pending.append((sample, chunks, fingerprint, cache_path))

for start in range(0, len(unpersisted_existing), CACHE_SHARD_SIZE):
    persist_cache_paths(unpersisted_existing[start:start + CACHE_SHARD_SIZE])

print(f"valid cached: {len(samples) - len(pending)}; pending: {len(pending)}")
if pending:
    capability_major = torch.cuda.get_device_capability(0)[0]
    qwen_dtype = torch.bfloat16 if capability_major >= 8 else torch.float16
    torch.cuda.reset_peak_memory_stats()
    convnext = ConvNeXtTinyFeatureExtractor(device="auto")
    qwen = FrozenQwenRepresentationExtractor(
        model_id=QWEN_MODEL_ID,
        revision=QWEN_MODEL_REVISION,
        device_map="auto",
        min_pixels=DEFAULT_MIN_PIXELS,
        max_pixels=DEFAULT_MAX_PIXELS,
        dtype=qwen_dtype,
    )
    shard_paths = []
    started_at = time.monotonic()
    for index, (sample, chunks, fingerprint, cache_path) in enumerate(pending, 1):
        frames = load_sample_images(sample)
        try:
            convnext_features = convnext.extract_frame_features(frames).global_features
            representation = build_structured_representation(
                sample=sample,
                frames=frames,
                convnext_frame_features=convnext_features,
                chunks=chunks,
                provider=qwen,
                fingerprint=fingerprint,
            )
            save_representation(cache_path, representation)
            shard_paths.append(cache_path)
        finally:
            for frame in frames:
                frame.close()
        if len(shard_paths) >= CACHE_SHARD_SIZE:
            persist_cache_paths(shard_paths)
            shard_paths = []
        if index % 10 == 0 or index == len(pending):
            elapsed = time.monotonic() - started_at
            print(f"[{index}/{len(pending)}] {elapsed / index:.2f} sec/sample", flush=True)
    persist_cache_paths(shard_paths)
    print("peak CUDA MB:", torch.cuda.max_memory_allocated() / 1024**2)
    del qwen, convnext
    gc.collect()
    torch.cuda.empty_cache()

final_cache_count = sum(1 for _ in LOCAL_CACHE_ROOT.rglob("*.pt"))
assert final_cache_count == 9535, final_cache_count
cache_manifest = {
    "samples": final_cache_count,
    "model_id": QWEN_MODEL_ID,
    "revision": QWEN_MODEL_REVISION,
    "min_pixels": DEFAULT_MIN_PIXELS,
    "max_pixels": DEFAULT_MAX_PIXELS,
}
with (DRIVE_CACHE_SHARDS / "manifest.json").open("w", encoding="utf-8") as file:
    json.dump(cache_manifest, file, indent=2)
    file.write("\n")
print(cache_manifest)


valid cached: 9535; pending: 0
{'samples': 9535, 'model_id': 'Qwen/Qwen2.5-VL-3B-Instruct', 'revision': '37eefda6797047d24d5d08184aea3e097f3f8a50', 'min_pixels': 100352, 'max_pixels': 200704}


## 8. Define full-data five-fold training

Every fold contains all 24 classes. Checkpoints and summaries are written directly to Drive.

In [8]:
from snu_ordering.structured_scorer import LightweightStructuredScorer
from snu_ordering.structured_training import (
    StructuredTrainingConfig,
    prepare_structured_samples,
    train_prepared_structured_scorer,
)

DEVICE = torch.device("cuda")
torch.set_float32_matmul_precision("high")
training_config = StructuredTrainingConfig(
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    batch_size=BATCH_SIZE,
    evaluation_batch_size=EVALUATION_BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    seed=SEED,
    top_h=TOP_H,
    top_m=TOP_M,
    alpha=ALPHA,
    beta=BETA,
    gamma=GAMMA,
    delta=DELTA,
    use_amp=True,
)

prepared_samples = prepare_structured_samples(
    samples, LOCAL_CACHE_ROOT, DEVICE, top_h=TOP_H
)

def seed_everything(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def run_fold(fold):
    output_dir = DRIVE_RESULTS_ROOT / f"fold_{fold}"
    if (output_dir / "training_summary.json").is_file():
        print(f"fold {fold}: already completed, skipping")
        return
    train_samples = [item for item in prepared_samples if item.sample.fold != fold]
    validation_samples = [item for item in prepared_samples if item.sample.fold == fold]
    assert {item.sample.rank_label for item in train_samples} == set(answer_counts)
    assert {item.sample.rank_label for item in validation_samples} == set(answer_counts)
    seed_everything(SEED)
    scorer = LightweightStructuredScorer(
        hidden_dim=HIDDEN_DIM,
        verifier=VERIFIER,
    ).to(DEVICE)
    print(f"fold {fold}: train={len(train_samples)}, validation={len(validation_samples)}")
    train_prepared_structured_scorer(
        scorer,
        train_samples,
        validation_samples,
        output_dir,
        training_config,
    )
    del scorer
    gc.collect()
    torch.cuda.empty_cache()


prepared candidates: 500/9535
prepared candidates: 1000/9535
prepared candidates: 1500/9535
prepared candidates: 2000/9535
prepared candidates: 2500/9535
prepared candidates: 3000/9535
prepared candidates: 3500/9535
prepared candidates: 4000/9535
prepared candidates: 4500/9535
prepared candidates: 5000/9535
prepared candidates: 5500/9535
prepared candidates: 6000/9535
prepared candidates: 6500/9535
prepared candidates: 7000/9535
prepared candidates: 7500/9535
prepared candidates: 8000/9535
prepared candidates: 8500/9535
prepared candidates: 9000/9535
prepared candidates: 9500/9535
prepared candidates: 9535/9535
resident training data: features=1.13 GiB, candidate_plans=0.35 GiB on cuda


## 9. Run selected folds

This cell is long-running. To parallelize, use separate runtimes with disjoint `FOLDS_TO_RUN` values.

In [9]:
for fold in FOLDS_TO_RUN:
    run_fold(fold)


fold 0: train=7620, validation=1915


/content/snu_ai/snu_ordering/structured_training.py:376: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_fp16_scaler)


{"epoch": 1, "train_loss": 2.9089466014872074, "samples": 1915, "exact_match_accuracy": 0.13890339425587467, "lightweight_top8_recall": 0.6720626631853786}
{"epoch": 2, "train_loss": 2.746479530409565, "samples": 1915, "exact_match_accuracy": 0.19738903394255874, "lightweight_top8_recall": 0.7101827676240209}
{"epoch": 3, "train_loss": 2.531337002256098, "samples": 1915, "exact_match_accuracy": 0.21148825065274152, "lightweight_top8_recall": 0.7096605744125326}
{"epoch": 4, "train_loss": 2.357059998399629, "samples": 1915, "exact_match_accuracy": 0.22506527415143604, "lightweight_top8_recall": 0.7049608355091384}
{"epoch": 5, "train_loss": 2.2098086602418756, "samples": 1915, "exact_match_accuracy": 0.22297650130548302, "lightweight_top8_recall": 0.6992167101827677}
{"epoch": 6, "train_loss": 2.084369509864667, "samples": 1915, "exact_match_accuracy": 0.25326370757180156, "lightweight_top8_recall": 0.7154046997389034}
{"epoch": 7, "train_loss": 1.945868491125232, "samples": 1915, "exac

## 10. Aggregate five-fold results and select the final epoch count

In [10]:
import statistics

fold_records = []
for fold in range(5):
    summary_path = DRIVE_RESULTS_ROOT / f"fold_{fold}/training_summary.json"
    if not summary_path.is_file():
        continue
    with summary_path.open("r", encoding="utf-8") as file:
        summary = json.load(file)
    best = max(summary["history"], key=lambda row: row["exact_match_accuracy"])
    fold_records.append({
        "fold": fold,
        "best_epoch": best["epoch"],
        "samples": best["samples"],
        "exact_match_accuracy": best["exact_match_accuracy"],
        "lightweight_top8_recall": best.get("lightweight_top8_recall"),
        "epochs_completed": summary["epochs_completed"],
    })

aggregate = {"folds": fold_records, "completed_fold_count": len(fold_records)}
if len(fold_records) == 5:
    total_validation_samples = sum(row["samples"] for row in fold_records)
    aggregate["total_validation_samples"] = total_validation_samples
    aggregate["weighted_exact_match_accuracy"] = sum(
        row["samples"] * row["exact_match_accuracy"] for row in fold_records
    ) / total_validation_samples
    selected_final_epochs = max(
        1, int(round(statistics.median(row["best_epoch"] for row in fold_records)))
    )
    aggregate["selected_final_epochs"] = selected_final_epochs
else:
    selected_final_epochs = None

with (DRIVE_RESULTS_ROOT / "five_fold_summary.json").open("w", encoding="utf-8") as file:
    json.dump(aggregate, file, indent=2)
    file.write("\n")
print(json.dumps(aggregate, indent=2))


{
  "folds": [
    {
      "fold": 0,
      "best_epoch": 6,
      "samples": 1915,
      "exact_match_accuracy": 0.25326370757180156,
      "lightweight_top8_recall": 0.7154046997389034,
      "epochs_completed": 10
    },
    {
      "fold": 1,
      "best_epoch": 15,
      "samples": 1905,
      "exact_match_accuracy": 0.28188976377952757,
      "lightweight_top8_recall": 0.6981627296587927,
      "epochs_completed": 19
    },
    {
      "fold": 2,
      "best_epoch": 12,
      "samples": 1886,
      "exact_match_accuracy": 0.28366914103923646,
      "lightweight_top8_recall": 0.7120890774125133,
      "epochs_completed": 16
    },
    {
      "fold": 3,
      "best_epoch": 15,
      "samples": 1911,
      "exact_match_accuracy": 0.28623757195185767,
      "lightweight_top8_recall": 0.7142857142857143,
      "epochs_completed": 19
    },
    {
      "fold": 4,
      "best_epoch": 10,
      "samples": 1918,
      "exact_match_accuracy": 0.291970802919708,
      "lightweight_top8_rec

## 11. Train one final scorer on all 9,535 rows

The epoch count is the median best epoch from MD5 group-stratified five-fold validation. This produces one final model, not an ensemble.

In [11]:
from dataclasses import replace

if TRAIN_FINAL_SCORER:
    if selected_final_epochs is None:
        raise RuntimeError("All five folds must finish before final training.")
    final_output_dir = DRIVE_RESULTS_ROOT / "final_scorer"
    final_summary_path = final_output_dir / "training_summary.json"
    if final_summary_path.is_file():
        print(f"final scorer already completed: {final_output_dir}")
    else:
        seed_everything(SEED)
        final_scorer = LightweightStructuredScorer(
            hidden_dim=HIDDEN_DIM,
            verifier=VERIFIER,
        ).to(DEVICE)
        final_config = replace(
            training_config,
            max_epochs=selected_final_epochs,
            patience=selected_final_epochs,
        )
        print(f"final training: samples={len(samples)}, epochs={selected_final_epochs}")
        train_prepared_structured_scorer(
            final_scorer,
            prepared_samples,
            [],
            final_output_dir,
            final_config,
        )
        del final_scorer
        gc.collect()
        torch.cuda.empty_cache()


final training: samples=9535, epochs=12
{"epoch": 1, "train_loss": 2.883805760777178}
{"epoch": 2, "train_loss": 2.674860150837811}
{"epoch": 3, "train_loss": 2.4641701926844997}
{"epoch": 4, "train_loss": 2.3079057611216407}
{"epoch": 5, "train_loss": 2.1728343395269163}
{"epoch": 6, "train_loss": 2.0566886602300714}
{"epoch": 7, "train_loss": 1.9317010005356818}
{"epoch": 8, "train_loss": 1.8447021560138848}
{"epoch": 9, "train_loss": 1.745712486798687}
{"epoch": 10, "train_loss": 1.6347009121340235}
{"epoch": 11, "train_loss": 1.5367250161953603}
{"epoch": 12, "train_loss": 1.4573193831911373}


## 12. Checks and next steps

Expected persistent outputs:

- `MyDrive/snu ai/cache/full_train_9535_shards/`: resumable frozen feature shards
- `MyDrive/snu ai/results/full_24class_group_stratified_v1/fold_*/best.pt`: five validation checkpoints
- `MyDrive/snu ai/results/full_24class_group_stratified_v1/five_fold_summary.json`: full 24-class CV result
- `MyDrive/snu ai/results/full_24class_group_stratified_v1/final_scorer/final.pt`: single final scorer trained on all 9,535 rows

The final inference pipeline must still be validated offline on one RTX 3090 and must not combine the five fold checkpoints.